# Reto 1 · Paso 1: Juntar y limpiar los datos
**Monitoreo de Saldo de Inventarios — FEMSA Salud Chile (FCV)**

**Objetivo de este notebook:** pasar de **14 archivos de Excel sueltos** (7 Kardex + 7 Balanzas) a **2 tablas limpias** listas para las alertas del paso 2:

| Tabla de salida | Qué es (ejemplo de Don Pepe) | Filas |
|---|---|---|
| `kardex_limpio.csv` | La lista del encargado de bodega, **todos los meses juntos** | ~107,000 (producto × mes) |
| `gl_existencias.csv` | Lo que dice el contador en la cuenta *Existencias*, **un renglón por mes** | 7 |

**Qué hacemos, en orden:**
1. Configurar la carpeta de los datos
2. Leer y juntar los 7 Kardex (quitando filas basura)
3. Leer las 7 Balanzas y sacar la cuenta `CTA_000184 EXISTENCIAS`
4. Recalcular la antigüedad nosotros mismos y marcar datos raros
5. Comprobar que julio cuadra con la propuesta del socio
6. Guardar las tablas limpias

## 0. Configuración

Este notebook sigue la estructura del repo:

```
RETO_MA2004B/
├── data/raw/          ← aquí van los Excel originales (Kardex y Balanzas)
├── data/processed/    ← aquí este notebook guarda las tablas limpias
└── notebooks/         ← aquí vive este notebook
```

Antes de correrlo, los Excel deben estar **desencriptados** en `data/raw/` (`dvc pull -r storage` + `./scripts/dvc_decrypt.sh`).
No importa si los nombres traen algo extra: el código busca el **mes** (`202601`, `202602`, …) dentro del nombre.

> ⚠️ Antes de hacer commit: **Clear All Outputs**. Las salidas muestran datos del inventario sin encriptar.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)

def raiz_del_repo():
    # Sube carpetas desde donde corre el notebook hasta encontrar pyproject.toml
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "pyproject.toml").exists():
            return carpeta
    raise FileNotFoundError("No encontré pyproject.toml: abre el notebook dentro del repo RETO_MA2004B")

ROOT = raiz_del_repo()
DATA_DIR = ROOT / "data" / "raw"         # Excel originales (desencriptados)
OUT_DIR = ROOT / "data" / "processed"    # tablas limpias que genera este notebook
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo:", ROOT)
print("Leyendo de:", DATA_DIR)
print("Guardando en:", OUT_DIR)

CUENTA_EXISTENCIAS = "CTA_000184"   # la cuenta 111080001 del kickoff, anonimizada en la Balanza

def mes_del_nombre(path):
    # Saca '202601' del nombre del archivo
    m = re.search(r"(20\d{2}(0[1-9]|1[0-2]))", path.name)
    return m.group(1) if m else None

## 1. Encontrar los archivos

Buscamos los Kardex (`INV_Kardex_AAAAMM`) y las Balanzas. Ojo: la de julio viene escrita **"Blalanza"**, por eso el patrón acepta las dos formas.

In [ ]:
archivos_kardex = sorted(DATA_DIR.glob("*Kardex_*.xlsx"), key=mes_del_nombre)
archivos_balanza = sorted(
    [p for p in DATA_DIR.glob("*.xlsx") if re.search(r"B(a|la)lanza", p.name, re.I)],
    key=mes_del_nombre,
)

print("Kardex encontrados:")
for p in archivos_kardex:
    print("  ", mes_del_nombre(p), "→", p.name)
print("\nBalanzas encontradas:")
for p in archivos_balanza:
    print("  ", mes_del_nombre(p), "→", p.name)

assert len(archivos_kardex) > 0, "No hay Kardex en data/raw/. ¿Corriste dvc pull y dvc_decrypt.sh?"
assert len(archivos_kardex) == len(archivos_balanza), "Debe haber un Kardex por cada Balanza"

## 2. Leer y juntar los 7 Kardex

Cada Kardex es la **foto de la bodega** al cierre de un mes. Los leemos todos y los apilamos uno debajo del otro, agregando la columna `MES`.

**Limpieza:** quitamos las filas **sin código de producto**. ¿Por qué? El Kardex de julio trae al final 4 filas de **totales ya sumados**. Si las dejamos, al sumar el inventario de julio saldría **3 veces más grande** de lo real.

In [ ]:
COLUMNAS_KARDEX = {
    "PERIODO": "PERIODO",
    "CODIGO": "CODIGO",
    "DESCRIPTOR": "DESCRIPTOR",
    "STOCK": "STOCK",
    "COSTO": "COSTO",
    "VALORIZACION": "VALORIZACION",
    "FULCOM": "FULCOM",                     # Fecha de Última COMpra
    "UNID CONV": "UNID_CONV",
    "MESES ANTIG.": "MESES_ANTIG_ORIGINAL", # la guardamos solo para compararla
    "ALERTA ANTIG.": "SEMAFORO_ORIGINAL",
    "LINEA": "LINEA",
    "NOMBRE LINEA": "NOMBRE_LINEA",
    "POLITICA CJE AL 30SEP": "POLITICA_CANJE",  # P1 = con canje, P0 = sin canje
}

partes, bitacora = [], []
for path in archivos_kardex:
    mes = mes_del_nombre(path)
    df = pd.read_excel(path, sheet_name="DETALLE KARDEX")
    filas_leidas = len(df)
    basura = df[df["CODIGO"].isna()]                 # filas de totales / vacías
    df = df[df["CODIGO"].notna()].copy()
    df = df[list(COLUMNAS_KARDEX)].rename(columns=COLUMNAS_KARDEX)
    df.insert(0, "MES", mes)
    partes.append(df)
    bitacora.append({
        "MES": mes,
        "filas_leidas": filas_leidas,
        "filas_basura_quitadas": len(basura),
        "valor_basura_quitado": basura["VALORIZACION"].sum(),
        "filas_finales": len(df),
    })

kardex = pd.concat(partes, ignore_index=True)
pd.DataFrame(bitacora)

👆 Fíjate en julio (`202607`): se quitaron 4 filas que sumaban **~$234 mil millones**. Esa era la "basura" que inflaba el total.

### Arreglar tipos de dato

- `FULCOM` a fecha (sin la hora, que no nos sirve).
- `MES_FECHA`: el **último día del mes**, que es la fecha de corte contra la que medimos la antigüedad.

In [ ]:
kardex["FULCOM"] = pd.to_datetime(kardex["FULCOM"], errors="coerce").dt.normalize()
kardex["PERIODO"] = pd.to_datetime(kardex["PERIODO"])
kardex["FECHA_CORTE"] = kardex["PERIODO"] + pd.offsets.MonthEnd(0)
kardex["LINEA"] = kardex["LINEA"].astype("Int64")

print(kardex.dtypes)
print(f"\nTotal: {len(kardex):,} filas (producto × mes), {kardex['CODIGO'].nunique():,} productos distintos")

## 3. Leer las 7 Balanzas → cuenta de Existencias

La Balanza es el **libro del contador**: trae ~850 cuentas (caja, bancos, proveedores…). Solo nos interesa una: **`CTA_000184 EXISTENCIAS`**.

La hoja tiene 7 filas de encabezado decorativo, así que las saltamos. Las columnas que importan son las del bloque **Balance**:

> **Saldo = Balance Débito − Balance Crédito**

Si el saldo saliera **negativo**, sería un saldo de *naturaleza contraria* (un inventario no puede valer menos que cero). Eso lo revisamos en el paso 2.

También traemos `CTA_000194 Existencias PT Terceriza` solo como referencia, porque hay que preguntarle al socio si entra en la conciliación.

In [ ]:
COLS_BALANZA = ["CUENTA", "DESCRIPCION",
                "ALTAS_DEBITO", "ALTAS_CREDITO",
                "BALANCE_DEBITO", "BALANCE_CREDITO",
                "INV_DEBITO", "INV_CREDITO",
                "PG_DEBITO", "PG_CREDITO", "MOVIMIENTO_MES"]

filas_gl = []
for path in archivos_balanza:
    mes = mes_del_nombre(path)
    hoja = [h for h in pd.ExcelFile(path).sheet_names if h.strip().upper() == "FCV"][0]
    bal = pd.read_excel(path, sheet_name=hoja, header=None, skiprows=7)
    bal = bal.iloc[:, :11]
    bal.columns = COLS_BALANZA
    bal = bal[bal["CUENTA"].astype(str).str.startswith("CTA_")]

    for cuenta in [CUENTA_EXISTENCIAS, "CTA_000194"]:
        r = bal[bal["CUENTA"] == cuenta]
        assert len(r) == 1, f"{cuenta} no encontrada (o repetida) en {path.name}"
        r = r.iloc[0]
        filas_gl.append({
            "MES": mes,
            "CUENTA": cuenta,
            "DESCRIPCION": r["DESCRIPCION"],
            "BALANCE_DEBITO": r["BALANCE_DEBITO"],
            "BALANCE_CREDITO": r["BALANCE_CREDITO"],
            "SALDO_GL": r["BALANCE_DEBITO"] - r["BALANCE_CREDITO"],
        })

gl_todas = pd.DataFrame(filas_gl)
gl = gl_todas[gl_todas["CUENTA"] == CUENTA_EXISTENCIAS].reset_index(drop=True)
gl["PERIODO"] = pd.to_datetime(gl["MES"] + "01")
gl

In [ ]:
# Solo referencia: la otra cuenta de existencias (para preguntar al socio)
gl_todas[gl_todas["CUENTA"] == "CTA_000194"][["MES", "DESCRIPCION", "SALDO_GL"]]

## 4. Recalcular la antigüedad y marcar datos raros

### ¿Por qué no usamos la columna `MESES ANTIG.` que ya viene?
Porque está **mal calculada**. Mira el mismo producto, con la **misma fecha de última compra**, mes a mes:

In [ ]:
kardex.loc[kardex["CODIGO"] == "SKU_006155",
           ["MES", "STOCK", "FULCOM", "MESES_ANTIG_ORIGINAL", "SEMAFORO_ORIGINAL"]]

La antigüedad debería subir **+1 cada mes** (12, 13, 14, 15, 16, 17, 18), pero en junio **retrocede a 13**. Además, en enero aparece con 12 meses cuando del 15-may-2025 al 31-ene-2026 hay ~8½ meses.

### Nuestra regla (simple y explicable)
> **Días de antigüedad = Fecha de corte (fin de mes) − FULCOM**

Con días puedes aplicar **cualquier** semáforo después (0–30, 0–90, 0–6 meses…), sin rehacer nada. Eso se decide en el paso 2, cuando el socio confirme la política.

In [ ]:
kardex["DIAS_ANTIG"] = (kardex["FECHA_CORTE"] - kardex["FULCOM"]).dt.days
kardex["MESES_ANTIG"] = np.floor(kardex["DIAS_ANTIG"] / 30.44)   # meses completos aprox.

# Ejemplo: el mismo SKU, ahora sí sube parejo
kardex.loc[kardex["CODIGO"] == "SKU_006155",
           ["MES", "FULCOM", "MESES_ANTIG_ORIGINAL", "DIAS_ANTIG", "MESES_ANTIG"]]

### Banderas de calidad de datos
En lugar de **borrar** filas sospechosas (perderíamos dinero del inventario y la conciliación no cuadraría), las **marcamos** con banderas `True/False`. Así el auditor las ve en el tablero.

| Bandera | Qué significa | Por qué importa |
|---|---|---|
| `F_FECHA_FUTURA` | La última compra es **posterior** al cierre del mes | Imposible: daría antigüedad negativa. Hoy el Excel lo pinta de 🟢 verde → oculta riesgo |
| `F_SIN_FECHA` | No tiene FULCOM | No se puede medir antigüedad |
| `F_CONV_FALTANTE` | Unidad de conversión vacía (#N/A en Excel) | Error de captura |
| `F_CONV_DISTINTA_1` | Unidad de conversión ≠ 1 | Revisar que el costo esté en la unidad correcta |
| `F_DUPLICADO` | El mismo código aparece 2+ veces en el mismo mes | Contaría doble el inventario |

In [ ]:
kardex["F_FECHA_FUTURA"] = kardex["DIAS_ANTIG"] < 0
kardex["F_SIN_FECHA"] = kardex["FULCOM"].isna()
kardex["F_CONV_FALTANTE"] = kardex["UNID_CONV"].isna()
kardex["F_CONV_DISTINTA_1"] = kardex["UNID_CONV"].notna() & (kardex["UNID_CONV"] != 1)
kardex["F_DUPLICADO"] = kardex.duplicated(subset=["MES", "CODIGO"], keep=False)

banderas = [c for c in kardex.columns if c.startswith("F_")]
kardex["DATO_OK"] = ~kardex[banderas].any(axis=1)

# Resumen por mes: cuántas filas tiene cada bandera
resumen_calidad = kardex.groupby("MES")[banderas + ["DATO_OK"]].sum()
resumen_calidad["% FECHA FUTURA"] = (kardex.groupby("MES")["F_FECHA_FUTURA"].mean() * 100).round(1)
resumen_calidad

⚠️ **Hallazgo importante:** cerca del **70%** de las filas tienen fecha de última compra **en el futuro**. Esto hay que preguntárselo al socio antes de construir el semáforo y el modelo de Markov. Puede ser que al anonimizar los datos se hayan recorrido las fechas.

In [ ]:
# ¿Cuánto dinero está en filas con fecha futura?
kardex.groupby("MES").apply(
    lambda g: pd.Series({
        "valor_total": g["VALORIZACION"].sum(),
        "valor_fecha_futura": g.loc[g["F_FECHA_FUTURA"], "VALORIZACION"].sum(),
    })
).assign(pct=lambda t: t["valor_fecha_futura"] / t["valor_total"] * 100)

👆 En **dinero** es todavía más fuerte: **~98% del valor** del inventario está en productos con fecha futura. Si no se resuelve, el semáforo diría que casi todo el inventario está "verde" sin que realmente lo sepamos.

### Chequeo rápido: ¿la valorización cuadra con stock × costo?

In [ ]:
kardex["DIF_VALORIZACION"] = kardex["STOCK"] * kardex["COSTO"] - kardex["VALORIZACION"]
print("Filas donde stock × costo ≠ valorización (más de $1):",
      (kardex["DIF_VALORIZACION"].abs() > 1).sum())

Da 0. Esto es normal porque en el Excel la valorización **se calcula** como stock × costo; es un control "circular" (siempre cuadra por diseño). Lo dejamos documentado.

## 5. Comprobación: ¿julio cuadra con la propuesta del socio?

La hoja *Dashboard Inventario Julio* de la propuesta dice:
- **15,357** partidas en el Kardex
- Detalle Kardex **$116,975,118,671**
- Saldo GL **$117,495,620,727**

Si nuestros números coinciden, la limpieza funcionó. ✅

In [ ]:
jul = kardex[kardex["MES"] == "202607"]
gl_jul = gl.loc[gl["MES"] == "202607", "SALDO_GL"].iloc[0]

print(f"Partidas julio:        {len(jul):>20,}   (esperado 15,357)")
print(f"Valor Kardex julio:    {jul['VALORIZACION'].sum():>20,.0f}   (esperado 116,975,118,671)")
print(f"Saldo GL julio:        {gl_jul:>20,.0f}   (esperado 117,495,620,727)")

assert len(jul) == 15_357
assert round(jul["VALORIZACION"].sum()) == 116_975_118_671
assert gl_jul == 117_495_620_727
print("\n✅ Todo cuadra con la propuesta")

### Vista previa del paso 2: bodega vs contador, mes a mes
Esto **todavía no es la alerta**, solo para que veas hacia dónde vamos. La diferencia de ~$500 millones cada mes coincide con el **"recetario"**, que no viene en el detalle del Kardex (pregunta para el socio).

In [ ]:
resumen = (kardex.groupby("MES")
           .agg(PRODUCTOS=("CODIGO", "nunique"), VALOR_KARDEX=("VALORIZACION", "sum"))
           .reset_index()
           .merge(gl[["MES", "SALDO_GL"]], on="MES"))
resumen["DIFERENCIA"] = resumen["SALDO_GL"] - resumen["VALOR_KARDEX"]
resumen["DIF_%"] = resumen["DIFERENCIA"] / resumen["SALDO_GL"] * 100
resumen["VAR_GL_VS_MES_ANT_%"] = resumen["SALDO_GL"].pct_change() * 100
resumen

## 6. Guardar las tablas limpias

Se guardan en `data/processed/` como CSV (con `utf-8-sig` para que Excel y Power BI lean bien los acentos).

> 🔒 Estos CSV **no se suben tal cual**: se encriptan con `./scripts/dvc_encrypt.sh` y se versionan con DVC (ver comandos en la guía).

In [ ]:
cols_salida = ["MES", "PERIODO", "FECHA_CORTE", "CODIGO", "DESCRIPTOR",
               "LINEA", "NOMBRE_LINEA", "POLITICA_CANJE",
               "STOCK", "COSTO", "VALORIZACION", "UNID_CONV",
               "FULCOM", "DIAS_ANTIG", "MESES_ANTIG",
               "MESES_ANTIG_ORIGINAL", "SEMAFORO_ORIGINAL",
               *banderas, "DATO_OK"]

kardex[cols_salida].to_csv(OUT_DIR / "kardex_limpio.csv", index=False, encoding="utf-8-sig")
gl.to_csv(OUT_DIR / "gl_existencias.csv", index=False, encoding="utf-8-sig")
resumen_calidad.to_csv(OUT_DIR / "resumen_calidad.csv", encoding="utf-8-sig")

for f in sorted(OUT_DIR.glob("*.csv")):
    print(f"data/processed/{f.name:25s} {f.stat().st_size/1e6:6.1f} MB")

## ✅ Resumen del paso 1

| Qué hicimos | Resultado |
|---|---|
| Juntamos 7 Kardex | 1 tabla de ~107 mil filas (producto × mes) |
| Quitamos basura | 4 filas de totales en julio (~$234 mil millones falsos) |
| Sacamos la cuenta de Existencias de 7 Balanzas | 1 tabla de 7 filas |
| Recalculamos antigüedad | `DIAS_ANTIG` consistente mes a mes |
| Marcamos datos raros | 5 banderas de calidad, sin borrar nada |
| Validamos contra la propuesta | Julio cuadra al peso ✅ |

### Preguntas pendientes para el socio
1. ¿Por qué ~70% de las fechas de última compra (FULCOM) son **posteriores** al cierre del mes?
2. ¿Qué semáforo es el oficial: 0–30 días / 0–90 días / 0–6 meses?
3. ¿De dónde sale el **recetario** (~$500 M de diferencia mensual) y nos pueden dar su detalle?
4. ¿`CTA_000194 Existencias PT Terceriza` entra en la conciliación?
5. ¿Reportamos en CLP o en MXN?

**Siguiente: Paso 2 → aplicar las 3 alertas** sobre `kardex_limpio.csv` y `gl_existencias.csv`.